## Task 1

In [ ]:
import re
from pathlib import Path



def read_file(filepath): #Leser en fil og returnerer (tekstinnhold, koding som ble brukt)
    path = Path(filepath)
    raw = path.read_bytes()

    try:
        # UTF-8 er streng og feiler hvis bytene ikke er gyldig UTF-8
        text = raw.decode("utf-8")
        encoding = "utf-8"
    except UnicodeDecodeError:
        # Latin-1 kan dekode alle bytes, så den brukes som reserve
        text = raw.decode("latin-1")
        encoding = "latin-1"

    return text, encoding



LOG_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>\w+)\] "
    r"item=(?P<item>.+?) "
    r"price=(?P<price>\d+(?:\.\d+)?) "
    r"(?P<currency>[A-Z]{3}) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
)


def parse_log(text):
    """Returnerer en liste med dictionaries, én per gyldig salgslinje."""
    records = []

    for line in text.splitlines():
        match = LOG_PATTERN.fullmatch(line.strip())
        if match is None:
            continue  # støylinje, tom linje eller ugyldig linje

        record = match.groupdict()
        record["price"] = float(record["price"])
        records.append(record)

    return records


def parse_log_file(filepath):
    text, _encoding = read_file(filepath)
    return parse_log(text)


if __name__ == "__main__":
    eksempel = (
        "--- PRINTER PAPER LOW WARNING ---\n"
        "[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)\n"
        "=== NETWORK SYNC OK ===\n"
        "[2026-10-15 08:31:12] [POS_02] item=Iced Latte price=55 NOK (REFUND)\n"
        "[2026-10-15 08:32:00] [POS_01] item=Mocha price=5.00 USD (PENDING)\n"
    )
    for rec in parse_log(eksempel):
        print(rec)

{'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 08:31:12', 'register_id': 'POS_02', 'item': 'Iced Latte', 'price': 55.0, 'currency': 'NOK', 'status': 'REFUND'}


## Task 2

In [ ]:
import json 
from pathlib import Path

def find_files(dictonary):
    dictonary = Path(dictonary)
    log_files = []

    for item in dictonary.iterdir(): 
        if items.is_dir():
            log_files.extend(find_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files

def find_active_registers(data) -> set[set]: 
   active_id = set()

   if insatnce(data, dict):
        if data.get("active") is True and "id" in data:
            active_id.add(data["id"])

        for data in data.values():
            active_ids |= find_active_registers(data)
        
        elif isinstance(data, list):
            for item in data:
                active_ids |= find_active_registers(item)

        return active_ids


def load_active_registers(config_path="data/store_config.json") -> set[str]:
    path = Path(config_path)
    with path.open(encoding="utf-8") as f:
        config = json.load(f)
    return find_active_registers(config)


if __name__ == "__main__":
    eksempel = {
        "company": "Campus Coffee",
        "regions": [
            {
                "region_name": "Main Campus",
                "stores": [
                    {
                        "name": "Library Cafe",
                        "registers": [
                            {"id": "POS_01", "active": True},
                            {"id": "POS_02", "active": False},
                        ],
                    }
                ],
            }
        ],
    }
    print(find_active_registers(eksempel))

